In [1]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder


#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Här Slår vi ihop datasettet och lägger in allt i en dataframe med alla säsnger.
df = pd.concat([s0_2017, s0_2018,s0_2019,s0_2020,s0_2021,s0_2022,s0_2023,s0_2024,s0_2025,s0_2026,
               s1_2017, s1_2018,s1_2019,s1_2020,s1_2021,s1_2022,s1_2023,s1_2024,s1_2025,s1_2026
               ])

print(df.shape)

df.head()

df.info()

df.isna().sum()

df["FTR"].value_counts()

# Här skriver vi in de features vi tänker ska vara bra att använda.OBS! vi har tagit bort vår target FTR!!!
candidate_features = [
    "HS","AS","HST","AST",
    "HF","AF","HC","AC",
    "HY","AY","HR","AR",
    "B365H","B365D","B365A",
    "BWH","BWD","BWA",
    "IWH","IWD","IWA",
    "PSH","PSD","PSA",
    "WHH","WHD","WHA",
    "VCH","VCD","VCA",
    "MaxH","MaxD","MaxA",
    "AvgH","AvgD","AvgA",
    "B365>2.5","B365<2.5",
    "P>2.5","P<2.5",
    "Max>2.5","Max<2.5",
    "Avg>2.5","Avg<2.5",
    "AHh","B365AHH","B365AHA",
    "PAHH","PAHA",
    "MaxAHH","MaxAHA",
    "AvgAHH","AvgAHA",
    "B365CH","B365CD","B365CA",
    "BWCH","BWCD","BWCA",
    "IWCH","IWCD","IWCA",
    "PSCH","PSCD","PSCA",
    "WHCH","WHCD","WHCA",
    "VCCH","VCCD","VCCA",
    "MaxCH","MaxCD","MaxCA",
    "AvgCH","AvgCD","AvgCA",
    "B365C>2.5","B365C<2.5",
    "PC>2.5","PC<2.5",
    "MaxC>2.5","MaxC<2.5",
    "AvgC>2.5","AvgC<2.5",
    "AHCh","B365CAHH","B365CAHA",
    "PCAHH","PCAHA",
    "MaxCAHH","MaxCAHA",
    "AvgCAHH","AvgCAHA"
]

#Dessa har vi tagit bort från variablerna på grund av data lekage.
leakage_columns = ["FTHG","FTAG","HTHG","HTAG","HTR"]

#Dessa tar vi bort för descision tree kan inte hantera strings. OBS! Vi vill använda hometeam och Awayteam i framtden
string_columns = ["Div","Date","Time","HomeTeam","AwayTeam","Referee"]

# Ta bort alla NA values i data och inkluderar Hometeam och Awayteam
df_model = df[candidate_features + ["HomeTeam", "AwayTeam", "FTR"]].dropna()

df_model = pd.get_dummies(df_model, columns=["HomeTeam", "AwayTeam"], drop_first=False)

# Här delar vi sedan up modellen i X och Y delar
X_model = df_model.drop(columns=["FTR"])
Y_model = df_model["FTR"]


# Nu skapar vi vårt Decision tree för att se vilka variabler som vi vill använda oss av!
tree = DecisionTreeClassifier(max_depth = 5, random_state = 42)

tree.fit(X_model,Y_model)

# Efter detta vill vi hämta ut och se vilka features som har högst importance        
importance = pd.DataFrame({"Features":X_model.columns, "Importance":tree.feature_importances_})

# Här tar vi de viktigaste från descision tree och skriver ut de i fallande ordning.
importance_asc = importance.sort_values(by = "Importance",ascending=False)

#Skriver ut hur viktiga features är 
print(importance_asc)

#Skriver ut 10 viktigaste som vi nu ska använda oss av!
print(importance_asc.head(10))

# Försöker lägga ihop alla importances för hemma och borta lag
home_importance = importance[importance["Features"].str.startswith("HomeTeam_")]["Importance"].sum()
away_importance = importance[importance["Features"].str.startswith("AwayTeam_")]["Importance"].sum()


#Print för total importance
print("HomeTeam total importance:", home_importance.sum())
print("AwayTeam total importance:", away_importance.sum())

(9119, 180)
<class 'pandas.DataFrame'>
Index: 9119 entries, 0 to 439
Columns: 180 entries, Div to LBCA
dtypes: float64(170), int64(2), str(8)
memory usage: 12.6 MB
               Features  Importance
2                   HST    0.442011
3                   AST    0.349384
6                    HC    0.044423
53               B365CH    0.030002
68                 VCCH    0.027660
..                  ...         ...
193  AwayTeam_West Brom    0.000000
194   AwayTeam_West Ham    0.000000
195      AwayTeam_Wigan    0.000000
196     AwayTeam_Wolves    0.000000
197    AwayTeam_Wycombe    0.000000

[198 rows x 2 columns]
    Features  Importance
2        HST    0.442011
3        AST    0.349384
6         HC    0.044423
53    B365CH    0.030002
68      VCCH    0.027660
59      IWCH    0.016650
7         AC    0.015976
1         AS    0.012475
37  B365<2.5    0.009777
79    PC>2.5    0.009329
HomeTeam total importance: 0.0
AwayTeam total importance: 0.0


In [2]:
# ## Göra nytt dataset (Kör inte denna,,, kör den under)

# import pandas as pd
# from sklearn.preprocessing import LabelEncoder

# # Skapa datasetet och bestäm features
# df_new = pd.DataFrame({
#     "season": pd.Series(dtype="int8"),
#     "date": pd.Series(dtype="datetime64[ns]"),
#     "homeTeam": pd.Series(dtype="string"),
#     "awayTeam": pd.Series(dtype="string"),
#     "lastSeasonDivisions": pd.Series(dtype="int8"),
#     "winLoseLast5": pd.Series(dtype="int8"),
#     "result": pd.Series(dtype="int8")
# })


# #Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
# s0_2017 = pd.read_csv("E0_2017.csv")
# s0_2018 = pd.read_csv("E0_2018.csv")
# s0_2019 = pd.read_csv("E0_2019.csv")
# s0_2020 = pd.read_csv("E0_2020.csv")
# s0_2021 = pd.read_csv("E0_2021.csv")
# s0_2022 = pd.read_csv("E0_2022.csv")
# s0_2023 = pd.read_csv("E0_2023.csv")
# s0_2024 = pd.read_csv("E0_2024.csv")
# s0_2025 = pd.read_csv("E0_2025.csv")
# s0_2026 = pd.read_csv("E0_2026.csv")

# s1_2017 = pd.read_csv("E1_2017.csv")
# s1_2018 = pd.read_csv("E1_2018.csv")
# s1_2019 = pd.read_csv("E1_2019.csv")
# s1_2020 = pd.read_csv("E1_2020.csv")
# s1_2021 = pd.read_csv("E1_2021.csv")
# s1_2022 = pd.read_csv("E1_2022.csv")
# s1_2023 = pd.read_csv("E1_2023.csv")
# s1_2024 = pd.read_csv("E1_2024.csv")
# s1_2025 = pd.read_csv("E1_2025.csv")
# s1_2026 = pd.read_csv("E1_2026.csv")

# # Gör en lista med säsonger
# seasons = list();
# seasons.append(pd.concat([s0_2017, s1_2017]));
# seasons.append(pd.concat([s0_2018, s1_2018]));
# seasons.append(pd.concat([s0_2019, s1_2019]));
# seasons.append(pd.concat([s0_2020, s1_2020]));
# seasons.append(pd.concat([s0_2021, s1_2021]));
# seasons.append(pd.concat([s0_2022, s1_2022]));
# seasons.append(pd.concat([s0_2023, s1_2023]));
# seasons.append(pd.concat([s0_2024, s1_2024]));
# seasons.append(pd.concat([s0_2025, s1_2025]));
# seasons.append(pd.concat([s0_2026, s1_2026]));


# # lägg till saker i den ny df
# for seasonsNumber in range(len(seasons)):
#     if (seasonsNumber < 3):
#         continue;
#     totalI = len(df_new);
#     for i in range(len(seasons[seasonsNumber])):
#         # lägg till season
#         df_new.loc[i + totalI, "season"] = 2017 + seasonsNumber;
#         # lägg till datum
#         df_new.loc[i + totalI, "date"] = seasons[seasonsNumber].iloc[i]["Date"];
#         # lägg till home team
#         df_new.loc[i + totalI, "homeTeam"] = seasons[seasonsNumber].iloc[i]["HomeTeam"];
#         # lägg till away team
#         df_new.loc[i + totalI, "awayTeam"] = seasons[seasonsNumber].iloc[i]["AwayTeam"];
#         # lägg till result
#         result = seasons[seasonsNumber].iloc[i]["FTHG"] - seasons[seasonsNumber].iloc[i]["FTAG"];
#         if (result == 0):
#             df_new.loc[i + totalI, "result"] = 1; # LIKA
#         if (result < 0):
#             df_new.loc[i + totalI, "result"] = 2; # BORTAVINST
#         if (result > 0):
#             df_new.loc[i + totalI, "result"] = 0; # HEMMAVINST
#         # lägg till lastSeasonDivisions
#         homeTeamDiv = "E7";
#         awayTeamDiv = "E7";
#         for j in range(len(seasons[seasonsNumber - 1])):
#             if (seasons[seasonsNumber].iloc[i]["HomeTeam"] == seasons[seasonsNumber - 1].iloc[j]["HomeTeam"]):
#                 homeTeamDiv = seasons[seasonsNumber - 1].iloc[j]["Div"];
#                 break
#         for k in range(len(seasons[seasonsNumber - 1])):
#             if (seasons[seasonsNumber].iloc[i]["AwayTeam"] == seasons[seasonsNumber - 1].iloc[k]["AwayTeam"]):
#                 awayTeamDiv = seasons[seasonsNumber - 1].iloc[k]["Div"];
#                 break
#         if (homeTeamDiv == awayTeamDiv):
#             df_new.loc[i + totalI, "lastSeasonDivisions"] = 0;
#         if (homeTeamDiv == "E7"):
#             df_new.loc[i + totalI, "lastSeasonDivisions"] = -1;
#         if (awayTeamDiv == "E7"):
#             df_new.loc[i + totalI, "lastSeasonDivisions"] = 1;
#         if (homeTeamDiv == "E0" and awayTeamDiv == "E1"):
#             df_new.loc[i + totalI, "lastSeasonDivisions"] = 1;
#         if (homeTeamDiv == "E1" and awayTeamDiv == "E0"):
#             df_new.loc[i + totalI, "lastSeasonDivisions"] = -1;

# df_new.to_csv("df_new.csv", index=False);

In [3]:
import pandas as pd

# Create empty list to collect rows (MUCH faster than df.loc in loop)
rows = []

# Load seasons (same as before)
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

seasons = [
    pd.concat([s0_2017, s1_2017]),
    pd.concat([s0_2018, s1_2018]),
    pd.concat([s0_2019, s1_2019]),
    pd.concat([s0_2020, s1_2020]),
    pd.concat([s0_2021, s1_2021]),
    pd.concat([s0_2022, s1_2022]),
    pd.concat([s0_2023, s1_2023]),
    pd.concat([s0_2024, s1_2024]),
    pd.concat([s0_2025, s1_2025]),
    pd.concat([s0_2026, s1_2026]),
]

for season_idx in range(0, len(seasons)):

    current = seasons[season_idx]
    previous = seasons[season_idx - 1]

    prev_div_map = dict(zip(previous["HomeTeam"], previous["Div"]))

    for _, row in current.iterrows():

        home = row["HomeTeam"]
        away = row["AwayTeam"]

        diff = row["FTHG"] - row["FTAG"]
        if diff > 0:
            result = 0
        elif diff < 0:
            result = 2
        else:
            result = 1

        homeDiv = prev_div_map.get(home, "E7")
        awayDiv = prev_div_map.get(away, "E7")

        if homeDiv == awayDiv:
            lastSeason = 0
        elif homeDiv == "E7":
            lastSeason = -1
        elif awayDiv == "E7":
            lastSeason = 1
        elif homeDiv == "E0" and awayDiv == "E1":
            lastSeason = 1
        elif homeDiv == "E1" and awayDiv == "E0":
            lastSeason = -1
        else:
            lastSeason = 0

        rows.append({
            "season": 2017 + season_idx,
            "date": row["Date"],
            "homeTeam": home,
            "awayTeam": away,
            "lastSeasonDivisions": lastSeason,

            "winLoseLast3":0,"winLoseLast5":0,"winLoseLast10":0,"winLoseLast20":0,"winLoseLast38":0,"winLoseLast76":0,

            "homeShotsOnGoalLast3":0,"homeShotsOnGoalLast5":0,"homeShotsOnGoalLast10":0,"homeShotsOnGoalLast20":0,"homeShotsOnGoalLast38":0,"homeShotsOnGoalLast76":0,
            "awayShotsOnGoalLast3":0,"awayShotsOnGoalLast5":0,"awayShotsOnGoalLast10":0,"awayShotsOnGoalLast20":0,"awayShotsOnGoalLast38":0,"awayShotsOnGoalLast76":0,
            "deltaShotsOnGoalLast3":0,"deltaShotsOnGoalLast5":0,"deltaShotsOnGoalLast10":0,"deltaShotsOnGoalLast20":0,"deltaShotsOnGoalLast38":0,"deltaShotsOnGoalLast76":0,

            "homeShotsAgainstGoalLast3":0,"homeShotsAgainstGoalLast5":0,"homeShotsAgainstGoalLast10":0,"homeShotsAgainstGoalLast20":0,"homeShotsAgainstGoalLast38":0,"homeShotsAgainstGoalLast76":0,
            "awayShotsAgainstGoalLast3":0,"awayShotsAgainstGoalLast5":0,"awayShotsAgainstGoalLast10":0,"awayShotsAgainstGoalLast20":0,"awayShotsAgainstGoalLast38":0,"awayShotsAgainstGoalLast76":0,
            "deltaShotsAgainstGoalLast3":0,"deltaShotsAgainstGoalLast5":0,"deltaShotsAgainstGoalLast10":0,"deltaShotsAgainstGoalLast20":0,"deltaShotsAgainstGoalLast38":0,"deltaShotsAgainstGoalLast76":0,

            "homeCornersLast3":0,"homeCornersLast5":0,"homeCornersLast10":0,"homeCornersLast20":0,"homeCornersLast38":0,"homeCornersLast76":0,
            "awayCornersLast3":0,"awayCornersLast5":0,"awayCornersLast10":0,"awayCornersLast20":0,"awayCornersLast38":0,"awayCornersLast76":0,
            "deltaCornersLast3":0,"deltaCornersLast5":0,"deltaCornersLast10":0,"deltaCornersLast20":0,"deltaCornersLast38":0,"deltaCornersLast76":0,

            "homeTeamHomeFormLast3":0,"homeTeamHomeFormLast5":0,"homeTeamHomeFormLast10":0,"homeTeamHomeFormLast20":0,"homeTeamHomeFormLast38":0,"homeTeamHomeFormLast76":0,
            "awayTeamAwayFormLast3":0,"awayTeamAwayFormLast5":0,"awayTeamAwayFormLast10":0,"awayTeamAwayFormLast20":0,"awayTeamAwayFormLast38":0,"awayTeamAwayFormLast76":0,
            "deltaFormLast3":0,"deltaFormLast5":0,"deltaFormLast10":0,"deltaFormLast20":0,"deltaFormLast38":0,"deltaFormLast76":0,

            "homeTeamYellowLast3":0,"homeTeamYellowLast5":0,"homeTeamYellowLast10":0,"homeTeamYellowLast20":0,"homeTeamYellowLast38":0,"homeTeamYellowLast76":0,
            "awayTeamYellowLast3":0,"awayTeamYellowLast5":0,"awayTeamYellowLast10":0,"awayTeamYellowLast20":0,"awayTeamYellowLast38":0,"awayTeamYellowLast76":0,
            "deltaYellowLast3":0,"deltaYellowLast5":0,"deltaYellowLast10":0,"deltaYellowLast20":0,"deltaYellowLast38":0,"deltaYellowLast76":0,

            # Added Last3 and Last10 here
            "homeTeamRedLast3":0,"homeTeamRedLast5":0,"homeTeamRedLast10":0,"homeTeamRedLast20":0,"homeTeamRedLast38":0,"homeTeamRedLast76":0,
            "awayTeamRedLast3":0,"awayTeamRedLast5":0,"awayTeamRedLast10":0,"awayTeamRedLast20":0,"awayTeamRedLast38":0,"awayTeamRedLast76":0,
            "deltaRedLast3":0,"deltaRedLast5":0,"deltaRedLast10":0,"deltaRedLast20":0,"deltaRedLast38":0,"deltaRedLast76":0,

            "homeTeamFoulsLast3":0,"homeTeamFoulsLast5":0,"homeTeamFoulsLast10":0,"homeTeamFoulsLast20":0,"homeTeamFoulsLast38":0,"homeTeamFoulsLast76":0,
            "awayTeamFoulsLast3":0,"awayTeamFoulsLast5":0,"awayTeamFoulsLast10":0,"awayTeamFoulsLast20":0,"awayTeamFoulsLast38":0,"awayTeamFoulsLast76":0,
            "deltaFoulsLast3":0,"deltaFoulsLast5":0,"deltaFoulsLast10":0,"deltaFoulsLast20":0,"deltaFoulsLast38":0,"deltaFoulsLast76":0,

            "homeTeamGoalDiffLast3":0,"homeTeamGoalDiffLast5":0,"homeTeamGoalDiffLast10":0,"homeTeamGoalDiffLast20":0,"homeTeamGoalDiffLast38":0,"homeTeamGoalDiffLast76":0,
            "awayTeamGoalDiffLast3":0,"awayTeamGoalDiffLast5":0,"awayTeamGoalDiffLast10":0,"awayTeamGoalDiffLast20":0,"awayTeamGoalDiffLast38":0,"awayTeamGoalDiffLast76":0,
            "deltaGoalDiffLast3":0,"deltaGoalDiffLast5":0,"deltaGoalDiffLast10":0,"deltaGoalDiffLast20":0,"deltaGoalDiffLast38":0,"deltaGoalDiffLast76":0,

            "result": result
        })

df_new = pd.DataFrame(rows)
df_new.to_csv("df_new_copy.csv", index=False)

In [4]:
## Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new_copy.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
results = df_new["result"].values

# Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5
n = 3;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    winLoseCounterHome = 0;
    iCounter = i-1;
    homeTeamResult = 0;
    while (winLoseCounterHome < (n+1) and iCounter >= 0):
        if (homeTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        if (awayTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        iCounter = iCounter -1;
    awayTeamResult = 0;
    iCounter = i-1;
    winLoseCounterAway = 0;
    while (winLoseCounterAway < (n+1) and iCounter >= 0):
        if (awayTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        if (homeTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "winLoseLast3"] = homeTeamResult - awayTeamResult;
    i = i - 1;
   
        
                        
df_new.to_csv("df_new2.csv", index=False);

In [5]:
## Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
results = df_new["result"].values

# Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5
n = 5;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    winLoseCounterHome = 0;
    iCounter = i-1;
    homeTeamResult = 0;
    while (winLoseCounterHome < (n+1) and iCounter >= 0):
        if (homeTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        if (awayTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        iCounter = iCounter -1;
    awayTeamResult = 0;
    iCounter = i-1;
    winLoseCounterAway = 0;
    while (winLoseCounterAway < (n+1) and iCounter >= 0):
        if (awayTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        if (homeTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "winLoseLast5"] = homeTeamResult - awayTeamResult;
    i = i - 1;
   
        
                        
df_new.to_csv("df_new2.csv", index=False);


In [6]:
## Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
results = df_new["result"].values

# Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5
n = 10;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    winLoseCounterHome = 0;
    iCounter = i-1;
    homeTeamResult = 0;
    while (winLoseCounterHome < (n+1) and iCounter >= 0):
        if (homeTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        if (awayTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        iCounter = iCounter -1;
    awayTeamResult = 0;
    iCounter = i-1;
    winLoseCounterAway = 0;
    while (winLoseCounterAway < (n+1) and iCounter >= 0):
        if (awayTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        if (homeTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "winLoseLast10"] = homeTeamResult - awayTeamResult;
    i = i - 1;
   
        
                        
df_new.to_csv("df_new2.csv", index=False);

In [7]:
## Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
results = df_new["result"].values

# Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5
n = 20;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    winLoseCounterHome = 0;
    iCounter = i-1;
    homeTeamResult = 0;
    while (winLoseCounterHome < (n+1) and iCounter >= 0):
        if (homeTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        if (awayTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        iCounter = iCounter -1;
    awayTeamResult = 0;
    iCounter = i-1;
    winLoseCounterAway = 0;
    while (winLoseCounterAway < (n+1) and iCounter >= 0):
        if (awayTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        if (homeTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "winLoseLast20"] = homeTeamResult - awayTeamResult;
    i = i - 1;
   
        
                        
df_new.to_csv("df_new2.csv", index=False);

In [8]:
## Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
results = df_new["result"].values

# Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5
n = 38;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    winLoseCounterHome = 0;
    iCounter = i-1;
    homeTeamResult = 0;
    while (winLoseCounterHome < (n+1) and iCounter >= 0):
        if (homeTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        if (awayTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        iCounter = iCounter -1;
    awayTeamResult = 0;
    iCounter = i-1;
    winLoseCounterAway = 0;
    while (winLoseCounterAway < (n+1) and iCounter >= 0):
        if (awayTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        if (homeTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "winLoseLast38"] = homeTeamResult - awayTeamResult;
    i = i - 1;
   
        
                        
df_new.to_csv("df_new2.csv", index=False);

In [9]:
## Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5

import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
results = df_new["result"].values

# Counts wins as +1 and losses as -1... then adds homeWin/losses - awayWin/Losses to column winLoseLast5
n = 76;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    winLoseCounterHome = 0;
    iCounter = i-1;
    homeTeamResult = 0;
    while (winLoseCounterHome < (n+1) and iCounter >= 0):
        if (homeTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        if (awayTeams[iCounter] == homeTeam):
            if (results[iCounter] == 0):
                homeTeamResult = homeTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
            elif (results[iCounter] == 2):
                homeTeamResult = homeTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterHome = winLoseCounterHome + 1;
                continue;
        iCounter = iCounter -1;
    awayTeamResult = 0;
    iCounter = i-1;
    winLoseCounterAway = 0;
    while (winLoseCounterAway < (n+1) and iCounter >= 0):
        if (awayTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        if (homeTeams[iCounter] == awayTeam):
            if (results[iCounter] == 0):
                awayTeamResult = awayTeamResult + 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 1):
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
            elif (results[iCounter] == 2):
                awayTeamResult = awayTeamResult - 1;
                iCounter = iCounter - 1;
                winLoseCounterAway = winLoseCounterAway + 1;
                continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "winLoseLast76"] = homeTeamResult - awayTeamResult;
    i = i - 1;
   
        
                        
df_new.to_csv("df_new2.csv", index=False);

In [10]:
# Counts home and away shots on target last 5 games
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HST"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AST"];
        if pd.notna(val):
            awayShots.append(int(val));

# Counts every shot on target for the home team in last n games
n = 3;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (homeTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (homeTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "homeShotsOnGoalLast3"] = shots;
    i = i - 1;

# Counts every shot on target for the away team in last n games
n = 3;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (awayTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (awayTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "awayShotsOnGoalLast3"] = shots;
    df_new.loc[i, "deltaShotsOnGoalLast3"] = (df_new.iloc[i]["homeShotsOnGoalLast3"] - shots);
    i = i - 1;

df_new.to_csv("df_new2.csv", index=False);

In [11]:
# Counts home and away shots on target last 5 games
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HST"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AST"];
        if pd.notna(val):
            awayShots.append(int(val));

# Counts every shot on target for the home team in last n games
n = 5;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (homeTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (homeTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "homeShotsOnGoalLast5"] = shots;
    i = i - 1;

# Counts every shot on target for the away team in last n games
n = 5;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (awayTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (awayTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "awayShotsOnGoalLast5"] = shots;
    df_new.loc[i, "deltaShotsOnGoalLast5"] = (df_new.iloc[i]["homeShotsOnGoalLast5"] - shots);
    i = i - 1;

df_new.to_csv("df_new2.csv", index=False);
        

In [12]:
# Counts home and away shots on target last 5 games
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HST"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AST"];
        if pd.notna(val):
            awayShots.append(int(val));

# Counts every shot on target for the home team in last n games
n = 10;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (homeTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (homeTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "homeShotsOnGoalLast10"] = shots;
    i = i - 1;

# Counts every shot on target for the away team in last n games
n = 10;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (awayTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (awayTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "awayShotsOnGoalLast10"] = shots;
    df_new.loc[i, "deltaShotsOnGoalLast10"] = (df_new.iloc[i]["homeShotsOnGoalLast10"] - shots);
    i = i - 1;

df_new.to_csv("df_new2.csv", index=False);

In [13]:
# Counts home and away shots on target last 5 games
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HST"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AST"];
        if pd.notna(val):
            awayShots.append(int(val));

# Counts every shot on target for the home team in last n games
n = 20;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (homeTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (homeTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "homeShotsOnGoalLast20"] = shots;
    i = i - 1;

# Counts every shot on target for the away team in last n games
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (awayTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (awayTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "awayShotsOnGoalLast20"] = shots;
    df_new.loc[i, "deltaShotsOnGoalLast20"] = (df_new.iloc[i]["homeShotsOnGoalLast20"] - shots);
    i = i - 1;

df_new.to_csv("df_new2.csv", index=False);

In [14]:
# Counts home and away shots on target last 5 games
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HST"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AST"];
        if pd.notna(val):
            awayShots.append(int(val));

# Counts every shot on target for the home team in last n games
n = 38;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (homeTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (homeTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "homeShotsOnGoalLast38"] = shots;
    i = i - 1;

# Counts every shot on target for the away team in last n games
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (awayTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (awayTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "awayShotsOnGoalLast38"] = shots;
    df_new.loc[i, "deltaShotsOnGoalLast38"] = (df_new.iloc[i]["homeShotsOnGoalLast38"] - shots);
    i = i - 1;

df_new.to_csv("df_new2.csv", index=False);

In [15]:
# Counts home and away shots on target last 5 games
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HST"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AST"];
        if pd.notna(val):
            awayShots.append(int(val));

# Counts every shot on target for the home team in last n games
n = 76;
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (homeTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (homeTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "homeShotsOnGoalLast76"] = shots;
    i = i - 1;

# Counts every shot on target for the away team in last n games
i = len(results) -1;
while (i > -1):
    homeTeam = homeTeams[i];
    awayTeam = awayTeams[i];
    shotsCounter = 0;
    iCounter = i-1;
    shots = 0;
    while (shotsCounter < (n+1) and iCounter >= 0):
        if (awayTeam == homeTeams[iCounter]):
            shots = shots + homeShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        elif (awayTeam == awayTeams[iCounter]):
            shots = shots + awayShots[iCounter];
            iCounter = iCounter - 1;
            shotsCounter = shotsCounter + 1;
            continue;
        iCounter = iCounter - 1;
    df_new.loc[i, "awayShotsOnGoalLast76"] = shots;
    df_new.loc[i, "deltaShotsOnGoalLast76"] = (df_new.iloc[i]["homeShotsOnGoalLast76"] - shots);
    i = i - 1;

df_new.to_csv("df_new2.csv", index=False);

In [16]:
# Counts home and away corners last 5 game
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

nameArray = list();
nameArray.append(3);
nameArray.append(5);
nameArray.append(10);
nameArray.append(20);
nameArray.append(38);
nameArray.append(76);

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HC"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AC"];
        if pd.notna(val):
            awayShots.append(int(val));

gurgel = 0;
while (gurgel < len(nameArray)):
    

    # Counts every shot on target for the home team in last n games
    n = nameArray[gurgel];
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (homeTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (homeTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "homeCornersLast" + str(n)] = shots;
        i = i - 1;
    
    # Counts every shot on target for the away team in last n games
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (awayTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (awayTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "awayCornersLast" + str(n)] = shots;
        df_new.loc[i, "deltaCornersLast" + str(n)] = (df_new.iloc[i]["homeCornersLast" + str(n)] - shots);
        i = i - 1;
    gurgel = gurgel +1;

df_new.to_csv("df_new2.csv", index=False);

In [ ]:
# Counts shots against
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

nameArray = list();
nameArray.append(3);
nameArray.append(5);
nameArray.append(10);
nameArray.append(20);
nameArray.append(38);
nameArray.append(76);

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HST"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AST"];
        if pd.notna(val):
            awayShots.append(int(val));

gurgel = 0;
while (gurgel < len(nameArray)):
    

    # Counts every shot on target for the home team in last n games
    n = nameArray[gurgel];
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (homeTeam == homeTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (homeTeam == awayTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "homeShotsAgainstGoalLast" + str(n)] = shots;
        i = i - 1;
    
    # Counts every shot on target for the away team in last n games
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (awayTeam == homeTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (awayTeam == awayTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "awayShotsAgainstGoalLast" + str(n)] = shots;
        df_new.loc[i, "deltaShotsAgainstGoalLast" + str(n)] = (df_new.iloc[i]["homeShotsAgainstGoalLast" + str(n)] - shots);
        i = i - 1;
    gurgel = gurgel +1;

df_new.to_csv("df_new2.csv", index=False);

In [ ]:
# Counts home away team form
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
result = df_new["result"].values

nameArray = list();
nameArray.append(3);
nameArray.append(5);
nameArray.append(10);
nameArray.append(20);
nameArray.append(38);
nameArray.append(76);

gurgel = 0;
while (gurgel < len(nameArray)):
    

    # Counts every shot on target for the home team in last n games
    n = nameArray[gurgel];
    i = len(result) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (homeTeam == homeTeams[iCounter]):
                if (result[iCounter] == 0):
                    shots = shots + 1;
                    shotsCounter = shotsCounter + 1;
                    iCounter = iCounter - 1;
                    continue;
                if (result[iCounter] == 2):
                    shots = shots - 1;
                    shotsCounter = shotsCounter + 1;
                    iCounter = iCounter - 1;
                    continue;
                shotsCounter = shotsCounter + 1;
                iCounter = iCounter - 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "homeTeamHomeFormLast" + str(n)] = shots;
        i = i - 1;
    
    # Counts every shot on target for the away team in last n games
    i = len(result) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (awayTeam == awayTeams[iCounter]):
                if (result[iCounter] == 0):
                    shots = shots - 1;
                    shotsCounter = shotsCounter + 1;
                    iCounter = iCounter - 1;
                    continue;
                if (result[iCounter] == 2):
                    shots = shots + 1;
                    shotsCounter = shotsCounter + 1;
                    iCounter = iCounter - 1;
                    continue;
                shotsCounter = shotsCounter + 1;
                iCounter = iCounter - 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "awayTeamAwayFormLast" + str(n)] = shots;
        df_new.loc[i, "deltaFormLast" + str(n)] = (df_new.iloc[i]["homeTeamHomeFormLast" + str(n)] - shots);
        i = i - 1;
    gurgel = gurgel +1;

df_new.to_csv("df_new2.csv", index=False);

In [ ]:
# Counts yellow C
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

nameArray = list();
nameArray.append(3);
nameArray.append(5);
nameArray.append(10);
nameArray.append(20);
nameArray.append(38);
nameArray.append(76);

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HY"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AY"];
        if pd.notna(val):
            awayShots.append(int(val));

gurgel = 0;
while (gurgel < len(nameArray)):
    

    # Counts every shot on target for the home team in last n games
    n = nameArray[gurgel];
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (homeTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (homeTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "homeTeamYellowLast" + str(n)] = shots;
        i = i - 1;
    
    # Counts every shot on target for the away team in last n games
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (awayTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (awayTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "awayTeamYellowLast" + str(n)] = shots;
        df_new.loc[i, "deltaYellowLast" + str(n)] = (df_new.iloc[i]["homeTeamYellowLast" + str(n)] - shots);
        i = i - 1;
    gurgel = gurgel +1;

df_new.to_csv("df_new2.csv", index=False);

In [ ]:
# Counts red C
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

nameArray = list();
nameArray.append(3);
nameArray.append(5);
nameArray.append(10);
nameArray.append(20);
nameArray.append(38);
nameArray.append(76);

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HR"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AR"];
        if pd.notna(val):
            awayShots.append(int(val));

gurgel = 0;
while (gurgel < len(nameArray)):
    

    # Counts every shot on target for the home team in last n games
    n = nameArray[gurgel];
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (homeTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (homeTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "homeTeamRedLast" + str(n)] = shots;
        i = i - 1;
    
    # Counts every shot on target for the away team in last n games
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (awayTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (awayTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "awayTeamRedLast" + str(n)] = shots;
        df_new.loc[i, "deltaRedLast" + str(n)] = (df_new.iloc[i]["homeTeamRedLast" + str(n)] - shots);
        i = i - 1;
    gurgel = gurgel +1;

df_new.to_csv("df_new2.csv", index=False);

In [ ]:
# Counts Fouls
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values
homeShots = list();
awayShots = list();

nameArray = list();
nameArray.append(3);
nameArray.append(5);
nameArray.append(10);
nameArray.append(20);
nameArray.append(38);
nameArray.append(76);

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["HF"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["AF"];
        if pd.notna(val):
            awayShots.append(int(val));

gurgel = 0;
while (gurgel < len(nameArray)):
    

    # Counts every shot on target for the home team in last n games
    n = nameArray[gurgel];
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (homeTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (homeTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "homeTeamFoulsLast" + str(n)] = shots;
        i = i - 1;
    
    # Counts every shot on target for the away team in last n games
    i = len(results) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n+1) and iCounter >= 0):
            if (awayTeam == homeTeams[iCounter]):
                shots = shots + homeShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (awayTeam == awayTeams[iCounter]):
                shots = shots + awayShots[iCounter];
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "awayTeamFoulsLast" + str(n)] = shots;
        df_new.loc[i, "deltaFoulsLast" + str(n)] = (df_new.iloc[i]["homeTeamFoulsLast" + str(n)] - shots);
        i = i - 1;
    gurgel = gurgel +1;

df_new.to_csv("df_new2.csv", index=False);

In [ ]:
# Counts goaldiffs
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Läs in df från förra sectionen
df_new = pd.read_csv("df_new2.csv");

#Här laddar vi in all data från våra csv Filer E0 = Premier leuage, E1 = Championship
s0_2017 = pd.read_csv("E0_2017.csv")
s0_2018 = pd.read_csv("E0_2018.csv")
s0_2019 = pd.read_csv("E0_2019.csv")
s0_2020 = pd.read_csv("E0_2020.csv")
s0_2021 = pd.read_csv("E0_2021.csv")
s0_2022 = pd.read_csv("E0_2022.csv")
s0_2023 = pd.read_csv("E0_2023.csv")
s0_2024 = pd.read_csv("E0_2024.csv")
s0_2025 = pd.read_csv("E0_2025.csv")
s0_2026 = pd.read_csv("E0_2026.csv")

s1_2017 = pd.read_csv("E1_2017.csv")
s1_2018 = pd.read_csv("E1_2018.csv")
s1_2019 = pd.read_csv("E1_2019.csv")
s1_2020 = pd.read_csv("E1_2020.csv")
s1_2021 = pd.read_csv("E1_2021.csv")
s1_2022 = pd.read_csv("E1_2022.csv")
s1_2023 = pd.read_csv("E1_2023.csv")
s1_2024 = pd.read_csv("E1_2024.csv")
s1_2025 = pd.read_csv("E1_2025.csv")
s1_2026 = pd.read_csv("E1_2026.csv")

# Gör en lista med säsonger
seasons = list();
seasons.append(pd.concat([s0_2017, s1_2017]));
seasons.append(pd.concat([s0_2018, s1_2018]));
seasons.append(pd.concat([s0_2019, s1_2019]));
seasons.append(pd.concat([s0_2020, s1_2020]));
seasons.append(pd.concat([s0_2021, s1_2021]));
seasons.append(pd.concat([s0_2022, s1_2022]));
seasons.append(pd.concat([s0_2023, s1_2023]));
seasons.append(pd.concat([s0_2024, s1_2024]));
seasons.append(pd.concat([s0_2025, s1_2025]));
seasons.append(pd.concat([s0_2026, s1_2026]));

homeTeams = df_new["homeTeam"].values
awayTeams = df_new["awayTeam"].values

homeShots = list();
awayShots = list();

nameArray = list();
nameArray.append(3);
nameArray.append(5);
nameArray.append(10);
nameArray.append(20);
nameArray.append(38);
nameArray.append(76);

for i in range(len(seasons)):
    for j in range(len(seasons[i])):
        val = seasons[i].iloc[j]["FTHG"];
        if pd.notna(val):
            homeShots.append(int(val));

for k in range(len(seasons)):
    for q in range(len(seasons[k])):
        val = seasons[k].iloc[q]["FTAG"];
        if pd.notna(val):
            awayShots.append(int(val));

gurgel = 0;
while (gurgel < len(nameArray)):
    

    # Counts every shot on target for the home team in last n games
    n = nameArray[gurgel];
    i = len(df_new) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n) and iCounter >= 0):
            if (homeTeam == homeTeams[iCounter]):
                shots = shots + (homeShots[iCounter] - awayShots[iCounter]);
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (homeTeam == awayTeams[iCounter]):
                shots = shots + (awayShots[iCounter] - homeShots[iCounter]);
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "homeTeamGoalDiffLast" + str(n)] = int(shots);
        i = i - 1;
    
    # Counts every shot on target for the away team in last n games
    i = len(df_new) -1;
    while (i > -1):
        homeTeam = homeTeams[i];
        awayTeam = awayTeams[i];
        shotsCounter = 0;
        iCounter = i-1;
        shots = 0;
        while (shotsCounter < (n) and iCounter >= 0):
            if (awayTeam == homeTeams[iCounter]):
                shots = shots + (homeShots[iCounter] - awayShots[iCounter]);
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            elif (awayTeam == awayTeams[iCounter]):
                shots = shots + (awayShots[iCounter] - homeShots[iCounter]);
                iCounter = iCounter - 1;
                shotsCounter = shotsCounter + 1;
                continue;
            iCounter = iCounter - 1;
        df_new.loc[i, "awayTeamGoalDiffLast" + str(n)] = int(shots);
        df_new.loc[i, "deltaGoalDiffLast" + str(n)] = int(df_new.iloc[i]["homeTeamGoalDiffLast" + str(n)] - shots);
        i = i - 1;
    gurgel = gurgel +1;

df_new.to_csv("df_new2.csv", index=False);

In [ ]:
# remove 2 first seasons

df_new = pd.read_csv("df_new2.csv");

df_final = df_new[~df_new["season"].isin([2017, 2018])];

df_final.to_csv("df_final.csv", index=False);

In [ ]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

df_new = pd.read_csv("df_final.csv");

# Här delar vi sedan up modellen i X och Y delar
X_model = df_new.drop(columns=["season", "date", "homeTeam", "awayTeam", "result"])
Y_model = df_new["result"]

# Nu skapar vi vårt Decision tree för att se vilka variabler som vi vill använda oss av!
tree = DecisionTreeClassifier(max_depth = 101, random_state = 42)

tree.fit(X_model,Y_model)

# Efter detta vill vi hämta ut och se vilka features som har högst importance        
importance = pd.DataFrame({"Features":X_model.columns, "Importance":tree.feature_importances_})

# Här tar vi de viktigaste från descision tree och skriver ut de i fallande ordning.
importance_asc = importance.sort_values(by = "Importance",ascending=False)

#Skriver ut hur viktiga features är 
print(importance_asc)

#Skriver ut 10 viktigaste som vi nu ska använda oss av!
print(importance_asc.head(50))


In [ ]:
#Börja med logisk regression 
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import LabelEncoder

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn import preprocessing

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.ensemble import AdaBoostClassifier

features = [
    "deltaGoalDiffLast20",
    "deltaCornersLast20",
    "homeCornersLast20",
    "deltaYellowLast76",
    "deltaShotsAgainstGoalLast38",
    "awayTeamYellowLast38",
    "deltaFormLast76",
    "awayShotsAgainstGoalLast20",
    "deltaCornersLast76",
    "deltaYellowLast20",
    "awayTeamFoulsLast76",
    "deltaFoulsLast5",
    "homeTeamFoulsLast20",
    "awayTeamGoalDiffLast20",
    "homeTeamGoalDiffLast76"
]

features2 = [
    "lastSeasonDivisions",

    "winLoseLast3",
    "winLoseLast5",
    "winLoseLast10",
    "winLoseLast20",
    "winLoseLast38",
    "winLoseLast76",
    
    "deltaShotsOnGoalLast3",
    "deltaShotsOnGoalLast5",
    "deltaShotsOnGoalLast10",
    "deltaShotsOnGoalLast20",
    "deltaShotsOnGoalLast38",
    "deltaShotsOnGoalLast76",
    
    "deltaShotsAgainstGoalLast3",
    "deltaShotsAgainstGoalLast5",
    "deltaShotsAgainstGoalLast10",
    "deltaShotsAgainstGoalLast20",
    "deltaShotsAgainstGoalLast38",
    "deltaShotsAgainstGoalLast76",
    
    "deltaCornersLast3",
    "deltaCornersLast5",
    "deltaCornersLast10",
    "deltaCornersLast20",
    "deltaCornersLast38",
    "deltaCornersLast76",
    
    "deltaFormLast3",
    "deltaFormLast5",
    "deltaFormLast10",
    "deltaFormLast20",
    "deltaFormLast38",
    "deltaFormLast76",
    
    "deltaYellowLast3",
    "deltaYellowLast5",
    "deltaYellowLast10",
    "deltaYellowLast20",
    "deltaYellowLast38",
    "deltaYellowLast76",
    
    "deltaRedLast3",
    "deltaRedLast5",
    "deltaRedLast10",
    "deltaRedLast20",
    "deltaRedLast38",
    "deltaRedLast76",
    
    "deltaFoulsLast3",
    "deltaFoulsLast5",
    "deltaFoulsLast10",
    "deltaFoulsLast20",
    "deltaFoulsLast38",
    "deltaFoulsLast76",
    
    "deltaGoalDiffLast3",
    "deltaGoalDiffLast5",
    "deltaGoalDiffLast10",
    "deltaGoalDiffLast20",
    "deltaGoalDiffLast38",
    "deltaGoalDiffLast76"
]

featuresHighDelta = [
    "deltaGoalDiffLast76",
    "deltaFormLast38",
    "deltaShotsAgainstGoalLast76",
    "deltaShotsOnGoalLast5",
    "winLoseLast20",
    "deltaYellowLast76",
    "lastSeasonDivisions",
    "deltaFoulsLast20",
    "deltaRedLast20",
    "deltaCornersLast10",
    ]

featuresTop10 = [
    "deltaGoalDiffLast76",
    "deltaGoalDiffLast5",
    "deltaFoulsLast76",
    "deltaRedLast10",
    "winLoseLast20",
    "deltaShotsAgainstGoalLast3",
    "deltaShotsOnGoalLast76",
    "deltaGoalDiffLast3",
    "deltaCornersLast38",
    "deltaShotsOnGoalLast5"
    ]

featuresLow = [
    "deltaGoalDiffLast3",
    "deltaFormLast3",
    "deltaShotsAgainstGoalLast3",
    "deltaShotsOnGoalLast3",
    "winLoseLast3",
    "deltaYellowLast3",
    "deltaFoulsLast3",
    "deltaRedLast3",
    "deltaCornersLast3",
    "deltaGoalDiffLast5",
    "deltaFormLast5",
    "deltaShotsAgainstGoalLast5",
    "deltaShotsOnGoalLast5",
    "winLoseLast5",
    "deltaYellowLast5",
    "lastSeasonDivisions",
    "deltaFoulsLast5",
    "deltaRedLast5",
    "deltaCornersLast5",
]

featuresHigh = [
    "deltaGoalDiffLast38",
    "deltaFormLast38",
    "deltaShotsAgainstGoalLast38",
    "deltaShotsOnGoalLast38",
    "winLoseLast38",
    "deltaYellowLast38",
    "deltaFoulsLast38",
    "deltaRedLast38",
    "deltaCornersLast38",
    "deltaGoalDiffLast76",
    "deltaFormLast76",
    "deltaShotsAgainstGoalLast76",
    "deltaShotsOnGoalLast76",
    "winLoseLast76",
    "deltaYellowLast76",
    "lastSeasonDivisions",
    "deltaFoulsLast76",
    "deltaRedLast76",
    "deltaCornersLast76",
]

scaler = preprocessing.MinMaxScaler();

featuresToUse = featuresTop10;

df_new = pd.read_csv("df_final.csv");

# Last 9 matches for prediction
df_pred = df_new.iloc[-11:]

#print(df_pred.head(11))

# All matches except the last 9 for training/testing
df_train_test = df_new.iloc[:-11]


X_model_LR = df_train_test[featuresToUse]
Y_model_LR = df_train_test["result"]

X_pred = df_pred[featuresToUse]

X_train,X_test,Y_train,Y_test = train_test_split(X_model_LR, Y_model_LR, test_size = 0.3)

X_train_unscaled = X_train
X_test_unscaled = X_test

Y_train_unscaled = Y_train
Y_test_unscaled = Y_test

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_pred = scaler.transform(X_pred)

model_LR = LogisticRegression(max_iter = 10000, C = 1);
model_SVM = SVC()
model_RF = RandomForestClassifier(n_estimators=200, max_depth=3, random_state=42)
model_AB = AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42)

model_SVM.fit(X_train,Y_train)
model_LR.fit(X_train,Y_train)
model_RF.fit(X_train_unscaled,Y_train_unscaled)
model_AB.fit(X_train_unscaled, Y_train_unscaled)




prediction_LR = model_LR.predict(X_train)
prediction_SVM = model_SVM.predict(X_train)
prediction_RF = model_RF.predict(X_train_unscaled)
prediction_AB = model_AB.predict(X_train_unscaled)

prediction_RF_Test = model_RF.predict(X_test_unscaled);
prediction_LR_Test = model_LR.predict(X_test)
prediction_SVM_Test = model_SVM.predict(X_test)
prediction_AB_Test = model_AB.predict(X_test_unscaled)

prediction_pred = model_LR.predict(X_pred)


print("Accuracy_LR = ", accuracy_score(Y_train, prediction_LR))
print("Accuracy_SVM = ", accuracy_score(Y_train, prediction_SVM))
print("Accuracy_RF = ", accuracy_score(Y_train_unscaled, prediction_RF))
print("Accuracy_AB = ", accuracy_score(Y_train_unscaled, prediction_AB))

print("Accuracy_LR_test = ", accuracy_score(Y_test, prediction_LR_Test))
print("Accuracy_SVM_test = ", accuracy_score(Y_test, prediction_SVM_Test))
print("Accuracy_RF_test = ", accuracy_score(Y_test_unscaled, prediction_RF_Test))
print("Accuracy_AB_test = ", accuracy_score(Y_test_unscaled, prediction_AB_Test))

#print("Importances = ",model.coef_)
#print(prediction_RF)
#print(prediction_LR)
#print(Y_train_unscaled)

importance = pd.DataFrame({
    "Feature": X_model_LR.columns,
    "Importance Absolute:": abs(model_LR.coef_).mean(axis=0)
})

importance = importance.sort_values(by="Importance Absolute:", ascending=False)

#print(importance)

importance = pd.DataFrame({
    "Feature": X_model_LR.columns,
    "Importance OG :": model_LR.coef_[0]
})

importance = importance.sort_values(by="Importance OG :", ascending=False)

#print(importance)

prob_pred = model_LR.predict_proba(X_pred)
print(prediction_pred)
print(prob_pred)

#print(df_pred["homeTeam"].iloc[0])

In [ ]:
import numpy as np

print("1 = Home Win, X = Draw, 2 = Away Win. ")


predictions = list()
for i in range(len(prediction_pred)):
    prediction = list()
    teams = df_pred["homeTeam"].iloc[i] + " vs " + df_pred["awayTeam"].iloc[i]
    res = int(prediction_pred[i])
    if(res== 0):
        res = "1";
    elif(res== 1):
        res = "X";
    elif(res== 2):
        res = "2";
        
    prediction.append(teams)
    prediction.append(res)
    predictions.append(prediction)
for i in range(len(predictions)):
    print("Match nr: ",i+1,predictions[i])

probGames = list()
for i in range(len(prediction_pred)):
    pred_class = prediction_pred[i]
    prob = float(prob_pred[i][pred_class])
    probGames.append(prob)


max_probs = np.max(prob_pred, axis=1)

lowest_6_indices = np.argsort(max_probs)[:6]

lowest_6_indices_sorted = sorted(lowest_6_indices)

lowest_6_indices_sorted = [int(i+1) for i in lowest_6_indices_sorted]

print("Add guards to: ",lowest_6_indices_sorted)

predictions_with_guards = []

for i in range(len(prediction_pred)):
    teams = df_pred["homeTeam"].iloc[i] + " vs " + df_pred["awayTeam"].iloc[i]
    main_res = int(prediction_pred[i])

    if(main_res== 0):
        main_res = "1";
    elif(main_res== 1):
        main_res = "X";
    elif(main_res == 2):
        main_res = "2"
    
    # Get the probabilities for this game
    probs = prob_pred[i]
    
   
    sorted_indices = np.argsort(probs)[::-1] 
    guard_res = sorted_indices[1]
    
    if(guard_res== 0):
        guard_res = "1";
    elif(guard_res== 1):
        guard_res = "X";
    elif(guard_res == 2):
        guard_res = "2"
    
    predictions_with_guards.append([teams, main_res, guard_res])

max_probs = np.max(prob_pred, axis=1)
lowest_6_indices = np.argsort(max_probs)[:6]

print("Games to add guards:")
for idx in lowest_6_indices:
    teams, main_res, guard_res = predictions_with_guards[idx]
    print(f"{teams}: Main={main_res}, Guard={guard_res}")

